# Bundestag API – Exploration
Direkte HTTP-Calls gegen `https://search.dip.bundestag.de/api/v1`

In [6]:
import requests
import json
import pprint

BASE_URL = "https://search.dip.bundestag.de/api/v1"
API_KEY  = "R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ"  # öffentlicher Demo-Key

session = requests.Session()
session.headers.update({
    "Authorization": f"ApiKey {API_KEY}",
    "Accept": "application/json",
})

print("Session bereit.")

Session bereit.


## Plenarprotokolle abrufen

In [7]:
url = f"{BASE_URL}/plenarprotokoll"

params = {
    "wahlperiode": 20,
    "format": "json",
    "num": 5,
    "datum.start": "2024-01-01",
}

response = session.get(url, params=params, timeout=15)
print(f"Status : {response.status_code}")
print(f"URL    : {response.url}")

Status : 200
URL    : https://search.dip.bundestag.de/api/v1/plenarprotokoll?wahlperiode=20&format=json&num=5&datum.start=2024-01-01


In [8]:
data = response.json()

print(f"Anzahl Dokumente : {len(data.get('documents', []))}")
print(f"Cursor (next)    : {data.get('cursor', '–')}")
print()
print("Antwort-Keys:", list(data.keys()))

Anzahl Dokumente : 100
Cursor (next)    : AoJwuJCB3pQDNFBsZW5hcnByb3Rva29sbC01Njk3

Antwort-Keys: ['numFound', 'documents', 'cursor']


In [9]:
# Erstes Dokument im Detail
erstes = data["documents"][0]
pprint.pprint(erstes)

{'aktualisiert': '2026-05-26T11:04:03+02:00',
 'datum': '2026-05-22',
 'dokumentart': 'Plenarprotokoll',
 'dokumentnummer': '21/81',
 'fundstelle': {'datum': '2026-05-22',
                'dokumentart': 'Plenarprotokoll',
                'dokumentnummer': '21/81',
                'herausgeber': 'BT',
                'id': '5796',
                'pdf_url': 'https://dserver.bundestag.de/btp/21/21081.pdf',
                'urheber': [],
                'verteildatum': '2026-05-26',
                'xml_url': 'https://dserver.bundestag.de/btp/21/21081.xml'},
 'herausgeber': 'BT',
 'id': '5796',
 'pdf_hash': 'f9536b0264e9f4d080d336ac305589c8',
 'titel': 'Protokoll der 81. Sitzung des 21. Deutschen Bundestages',
 'typ': 'Dokument',
 'vorgangsbezug': [{'id': '321621',
                    'titel': 'Bericht der Bundesregierung zum Stand der '
                             'Bemühungen um Rüstungskontrolle, Abrüstung und '
                             'Nichtverbreitung sowie über die Entwicklung 

In [10]:
# Alle Dokumente tabellarisch
for dok in data["documents"]:
    print(f"{dok.get('datum', '?'):12}  id={dok.get('id', '?'):<8}  {dok.get('titel', '')[:80]}")

2026-05-22    id=5796      Protokoll der 81. Sitzung des 21. Deutschen Bundestages
2026-05-21    id=5795      Protokoll der 80. Sitzung des 21. Deutschen Bundestages
2026-05-20    id=5794      Protokoll der 79. Sitzung des 21. Deutschen Bundestages
2026-05-08    id=5793      Protokoll der 1065. Sitzung des Bundesrates
2026-05-08    id=5792      Protokoll der 78. Sitzung des 21. Deutschen Bundestages
2026-05-07    id=5791      Protokoll der 77. Sitzung des 21. Deutschen Bundestages
2026-05-06    id=5790      Protokoll der 76. Sitzung des 21. Deutschen Bundestages
2026-04-24    id=5789      Protokoll der 1064. Sitzung des Bundesrates
2026-04-24    id=5788      Protokoll der 75. Sitzung des 21. Deutschen Bundestages
2026-04-23    id=5787      Protokoll der 74. Sitzung des 21. Deutschen Bundestages
2026-04-22    id=5786      Protokoll der 73. Sitzung des 21. Deutschen Bundestages
2026-04-17    id=5785      Protokoll der 72. Sitzung des 21. Deutschen Bundestages
2026-04-16    id=5784      P

## Weitere Endpunkte

In [11]:
def call_api(endpoint: str, **params) -> dict:
    """Hilfsfunktion: GET gegen einen beliebigen Endpunkt."""
    defaults = {"wahlperiode": 20, "format": "json", "num": 5}
    r = session.get(f"{BASE_URL}/{endpoint}", params={**defaults, **params}, timeout=15)
    r.raise_for_status()
    return r.json()

# Drucksachen
drucksachen = call_api("drucksache", **{"datum.start": "2024-01-01"})
print(f"Drucksachen: {len(drucksachen.get('documents', []))} gefunden")
for d in drucksachen["documents"]:
    print(f"  {d.get('datum', '?'):12}  {d.get('drucksachentyp', '?'):20}  {d.get('titel', '')[:60]}")

Drucksachen: 100 gefunden
  2026-05-26    ?                     zu dem Antrag der Abgeordneten Katharina Beck, Sandra Stein,
  2026-05-26    ?                     Entwurf eines Zweiten Gesetzes zur Änderung des Düngegesetze
  2026-05-26    ?                     Entwurf eines Gesetzes zur Durchführung der Verordnung (EU) 
  2026-05-26    ?                     Entwurf eines Gesetzes zur Änderung des Strafrechts - Umsetz
  2026-05-26    ?                     Entwurf eines Gesetzes zur Stärkung digitaler Ermittlungsbef
  2026-05-26    ?                     Entwurf eines Gesetzes zur Stärkung digitaler Ermittlungsbef
  2026-05-26    ?                     Entwurf eines Gesetzes zur Stabilisierung der Beitragssätze 
  2026-05-26    ?                     Entwurf eines Gesetzes zur Ermöglichung der digitalen Flugga
  2026-05-26    ?                     Entwurf eines Zweiten Gesetzes zur Änderung des Bundesbedarf
  2026-05-22    ?                     Entschließung des Bundesrates "Ausbau der dig

In [12]:
# Personen / MdBs
personen = call_api("person", **{"fraktionMitgliedschaft.fraktion": "SPD"})
print(f"SPD-MdBs: {len(personen.get('documents', []))} gefunden")
for p in personen["documents"]:
    print(f"  {p.get('nachname', '?')}, {p.get('vorname', '?')}")

SPD-MdBs: 100 gefunden
  Zobel, Vanessa
  Komning, Enrico
  Vriesema, Mayra
  Roth, Claudia
  Amthor, Philipp
  Klingbeil, Lars
  Schneider, Julia
  Schmidt, Julian
  Schmidt, Paul
  Schliesing, David
  Scheirich, Raimond
  Rottwilm, Philipp
  Rentzsch, Matthias
  Raue, Arne
  Mirow, Sahra
  Minich, Sergej
  Matzerath, Markus
  Martel, Johann
  Lensing, Sascha
  Ladzinski, Thomas
  Krieger, Lukas
  Köstering, Jan
  Körner, Konrad
  Korell, Thomas
  Köktürk, Cansin
  Koegel, Jürgen
  Kever, Rocco
  Kaminski, Maren
  Zerr, Anne
  Wiegelmann, Johannes
  Wagner, Sascha
  Volkmann, Johannes
  Valent, Aaron
  Dzienus, Timon
  Drößler, Christopher
  Dillschneider, Jeanne
  Conrad, Agnes
  Broßart, Victoria
  Pauli, Denis
  Pantisano, Luigi
  Böttger, Janina
  Bosch, Jorrit
  Teske, Robert
  Bohnhof, Peter
  Bock, Violetta
  Strauß, Otto
  Birghan, Christoph
  Bilic, Florian
  Becker, Desiree
  Bauer, Marcel
  Balten, Adam
  Hilmer, Olaf
  Heuberger, Moritz
  Hermeier, Mareike
  Henze, Stefan


---
## 💾 Alle Plenarprotokolle in SQLite speichern

Lädt **alle** verfügbaren Protokolle der Wahlperiode über Cursor-Pagination  
und speichert die Rohdaten in `data/plenarprotokolle.db`.

In [13]:
import sqlite3
import json
from pathlib import Path
from datetime import datetime

DB_PATH = Path("../../data/plenarprotokolle.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

conn.executescript("""
    CREATE TABLE IF NOT EXISTS protokolle (
        id            TEXT PRIMARY KEY,
        titel         TEXT,
        datum         TEXT,
        wahlperiode   INTEGER,
        sitzungsnr    INTEGER,
        pdf_url       TEXT,
        abstract      TEXT,
        raw_json      TEXT,          -- vollständiges Dokument als JSON
        gespeichert_am TEXT NOT NULL
    );

    CREATE INDEX IF NOT EXISTS idx_datum       ON protokolle(datum DESC);
    CREATE INDEX IF NOT EXISTS idx_wahlperiode ON protokolle(wahlperiode);
""")
conn.commit()
print(f"✅ Datenbank bereit: {DB_PATH.resolve()}")

✅ Datenbank bereit: C:\Users\janwe\Documents\Jan\Projects\PolitCheck_VS\data\plenarprotokolle.db


In [14]:
import time

def fetch_alle_protokolle(
    wahlperiode: int = 20,
    datum_von: str | None = None,
    datum_bis: str | None = None,
    batch_size: int = 100,
) -> list[dict]:
    """
    Lädt alle Plenarprotokolle einer Wahlperiode via Cursor-Pagination.

    Args:
        wahlperiode: Wahlperiode (default: 20)
        datum_von:   Startdatum YYYY-MM-DD (optional)
        datum_bis:   Enddatum   YYYY-MM-DD (optional)
        batch_size:  Dokumente pro API-Request (max 100)

    Returns:
        Liste aller Protokoll-Dicts
    """
    alle = []
    cursor = None
    seite = 1

    while True:
        params = {
            "wahlperiode": wahlperiode,
            "format": "json",
            "num": batch_size,
        }
        if datum_von:
            params["datum.start"] = datum_von
        if datum_bis:
            params["datum.end"] = datum_bis
        if cursor:
            params["cursor"] = cursor

        r = session.get(f"{BASE_URL}/plenarprotokoll", params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        batch = data.get("documents", [])
        alle.extend(batch)

        print(f"  Seite {seite:>3} → {len(batch):>3} Protokolle  (gesamt: {len(alle)})")

        cursor = data.get("cursor")
        if not batch or not cursor:
            break

        seite += 1
        time.sleep(0.3)   # höfliches Rate-Limiting

    return alle

print("Funktion 'fetch_alle_protokolle' geladen.")

Funktion 'fetch_alle_protokolle' geladen.


In [15]:
def speichere_protokolle(protokolle: list[dict], conn: sqlite3.Connection) -> dict:
    """
    Schreibt eine Liste von Protokoll-Dicts in die Datenbank.
    Bereits vorhandene IDs werden übersprungen (INSERT OR IGNORE).

    Returns:
        {"neu": int, "duplikate": int}
    """
    neu = 0
    duplikate = 0
    jetzt = datetime.now().isoformat()

    for dok in protokolle:
        doc_id = str(dok.get("id", ""))
        try:
            conn.execute(
                """INSERT OR IGNORE INTO protokolle
                   (id, titel, datum, wahlperiode, sitzungsnr,
                    pdf_url, abstract, raw_json, gespeichert_am)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                (
                    doc_id,
                    dok.get("titel"),
                    dok.get("datum"),
                    dok.get("wahlperiode"),
                    dok.get("dokumentnummer"),          # Sitzungsnummer
                    dok.get("fundstelle", {}).get("pdf_url"),
                    dok.get("abstract"),
                    json.dumps(dok, ensure_ascii=False),
                    jetzt,
                ),
            )
            if conn.execute("SELECT changes()").fetchone()[0] > 0:
                neu += 1
            else:
                duplikate += 1
        except Exception as e:
            print(f"  ⚠️  Fehler bei id={doc_id}: {e}")

    conn.commit()
    return {"neu": neu, "duplikate": duplikate}

print("Funktion 'speichere_protokolle' geladen.")

Funktion 'speichere_protokolle' geladen.


In [16]:
# ── Konfiguration ────────────────────────────────────────────────
WAHLPERIODE = 20
DATUM_VON   = "2021-10-01"   # Beginn der 20. Wahlperiode
DATUM_BIS   = None           # None = bis heute
# ─────────────────────────────────────────────────────────────────

print(f"📥 Lade alle Plenarprotokolle (WP {WAHLPERIODE}) ab {DATUM_VON} …\n")

protokolle = fetch_alle_protokolle(
    wahlperiode=WAHLPERIODE,
    datum_von=DATUM_VON,
    datum_bis=DATUM_BIS,
)

print(f"\n✅ {len(protokolle)} Protokolle geladen — speichere in DB …")
stats = speichere_protokolle(protokolle, conn)

print(f"\n📊 Ergebnis:")
print(f"   Neu gespeichert : {stats['neu']}")
print(f"   Bereits in DB   : {stats['duplikate']}")

📥 Lade alle Plenarprotokolle (WP 20) ab 2021-10-01 …

  Seite   1 → 100 Protokolle  (gesamt: 100)
  Seite   2 → 100 Protokolle  (gesamt: 200)
  Seite   3 → 100 Protokolle  (gesamt: 300)
  Seite   4 → 100 Protokolle  (gesamt: 400)
  Seite   5 → 100 Protokolle  (gesamt: 500)
  Seite   6 → 100 Protokolle  (gesamt: 600)
  Seite   7 → 100 Protokolle  (gesamt: 700)
  Seite   8 → 100 Protokolle  (gesamt: 800)
  Seite   9 → 100 Protokolle  (gesamt: 900)
  Seite  10 → 100 Protokolle  (gesamt: 1000)
  Seite  11 → 100 Protokolle  (gesamt: 1100)
  Seite  12 → 100 Protokolle  (gesamt: 1200)
  Seite  13 → 100 Protokolle  (gesamt: 1300)
  Seite  14 → 100 Protokolle  (gesamt: 1400)
  Seite  15 → 100 Protokolle  (gesamt: 1500)
  Seite  16 → 100 Protokolle  (gesamt: 1600)
  Seite  17 → 100 Protokolle  (gesamt: 1700)
  Seite  18 → 100 Protokolle  (gesamt: 1800)
  Seite  19 → 100 Protokolle  (gesamt: 1900)
  Seite  20 → 100 Protokolle  (gesamt: 2000)
  Seite  21 → 100 Protokolle  (gesamt: 2100)
  Seite  2

### 🔍 Datenbank abfragen

In [17]:
# Gesamtübersicht
total = conn.execute("SELECT COUNT(*) FROM protokolle").fetchone()[0]
aeltestes = conn.execute("SELECT MIN(datum) FROM protokolle").fetchone()[0]
neuestes  = conn.execute("SELECT MAX(datum) FROM protokolle").fetchone()[0]

print(f"Protokolle in DB : {total}")
print(f"Zeitraum         : {aeltestes}  →  {neuestes}")

# Letzte 10 Einträge
print("\nNeueste 10 Protokolle:")
rows = conn.execute("""
    SELECT datum, sitzungsnr, titel
    FROM protokolle
    ORDER BY datum DESC
    LIMIT 10
""").fetchall()
for r in rows:
    print(f"  {r['datum']:12}  Nr.{str(r['sitzungsnr'] or '?'):>5}  {(r['titel'] or '')[:70]}")

Protokolle in DB : 5779
Zeitraum         : 1949-09-07  →  2026-05-22

Neueste 10 Protokolle:
  2026-05-22    Nr.21/81  Protokoll der 81. Sitzung des 21. Deutschen Bundestages
  2026-05-21    Nr.21/80  Protokoll der 80. Sitzung des 21. Deutschen Bundestages
  2026-05-20    Nr.21/79  Protokoll der 79. Sitzung des 21. Deutschen Bundestages
  2026-05-08    Nr. 1065  Protokoll der 1065. Sitzung des Bundesrates
  2026-05-08    Nr.21/78  Protokoll der 78. Sitzung des 21. Deutschen Bundestages
  2026-05-07    Nr.21/77  Protokoll der 77. Sitzung des 21. Deutschen Bundestages
  2026-05-06    Nr.21/76  Protokoll der 76. Sitzung des 21. Deutschen Bundestages
  2026-04-24    Nr. 1064  Protokoll der 1064. Sitzung des Bundesrates
  2026-04-24    Nr.21/75  Protokoll der 75. Sitzung des 21. Deutschen Bundestages
  2026-04-23    Nr.21/74  Protokoll der 74. Sitzung des 21. Deutschen Bundestages


In [18]:
# Verbindung schließen wenn fertig
conn.close()
print("Verbindung geschlossen.")

Verbindung geschlossen.
